In [ ]:
import marimo as mo
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
enter_salary = mo.ui.slider(start=20, stop=120, step=1, value=50, label="Annual gross FTE salary (£k)")

In [ ]:
enter_years = mo.ui.slider(start=1, stop=45, step=1, value=10, label="Years of service (years)")

In [ ]:
enter_age = mo.ui.slider(start=20, stop=80, step=1, value=50, label="Age (years)")

In [ ]:
enter_frac = mo.ui.slider(start=0.2, stop=1, step=0.01, value=1, label="Fractional FTE ")

In [ ]:
def costs(years,salary,age,frac):

    # from sec 2 https://universityofsussex.sharepoint.com/sites/ChangeAndTransformation/SitePages/Voluntary-Redundancy.aspx
    vr_multiplier = 1.5
    age_multiplier = 0.5 if age<22 else 1 if age<41 else 1.5

    # NI is 15% of earnings over 5k
    ni = 0.15 * (salary-5)
    ni_with_frac = 0.15 * (frac*salary-5)

    # employer contrib to USS pension is 14.5%
    uss= 0.145 * salary

    annual_cost_to_uni = ni + uss + salary
    with_fractional = ni_with_frac + uss*frac + salary*frac

    VR_cost = vr_multiplier * age_multiplier * annual_cost_to_uni/52 * years

    # cap of £95K on total VR
    if VR_cost > 95:
        VR_cost = 95

    CR_cost = age_multiplier * annual_cost_to_uni/52 * years
    # cap of £22,530 on total CR   
    if CR_cost > 22.53:
        CR_cost = 22.53

    # 2 years min service requirement for stat redundancy pay
    if years < 2:
        CR_cost = 0

    return annual_cost_to_uni, with_fractional, VR_cost, CR_cost

In [ ]:
def cheaper_to_keep(salary=50, age=50, frac=1):

    years_of_service = np.linspace(1, age-18, age-18)

    cost_if_cr = [] # one-off cost compulsory redundancy
    cost_if_vr = [] # one-off cost
    cost_if_keep = [] # cost to keep for one year.
    cost_if_keep_frac = [] # cost to keep for one year at frac FTE

    for y in years_of_service:

        annual_cost_to_uni, with_fractional, VR_cost, CR_cost = costs(y,salary,age,frac)

        cost_if_keep.append(annual_cost_to_uni)
        cost_if_keep_frac.append(with_fractional)
        cost_if_vr.append(VR_cost)
        cost_if_cr.append(CR_cost)

    # Plot    
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4,4), layout='tight')
    ax.plot( years_of_service, cost_if_keep ,c='green', lw=3, alpha=0.6, label='Cost 1 year')
    ax.plot( years_of_service, cost_if_keep_frac ,c='seagreen', ls='--', lw=3, alpha=0.6, label=f'at {frac} FTE')
    ax.plot( years_of_service, cost_if_vr ,c='hotpink',ls='-', lw=3, alpha=0.6, label='Cost of VR')
    ax.plot( years_of_service, cost_if_cr ,c='red',ls=':', lw=3, alpha=0.6, label='Cost of CR')
    ax.set_xlabel('Years of service')
    ax.set_ylabel('Cost to University')
    ax.set_ylim(0,180)
    ax.grid()
    ax.legend(loc='upper center', ncols=2)
    return fig

In [ ]:
def slide():

    plot_output = mo.as_html(cheaper_to_keep(enter_salary.value, enter_age.value, enter_frac.value))

    title = mo.md("# Cheaper to keep?")

    text1= mo.md(
        f"""
        This takes into account NI, USS pension, and calcs from https://universityofsussex.sharepoint.com/sites/ChangeAndTransformation/SitePages/Voluntary-Redundancy.aspx
        """
    )

    text2 = mo.md(
        f"""
        Notice that the salary effect is the same for keep and lose.
        """
    )

    ss = mo.vstack(
    [ enter_salary,  enter_age, enter_frac],   
    justify="start",
    )

    stack=mo.hstack(
        [ 
        mo.vstack([ title, text1, ss, text2]),
        mo.vstack([ plot_output,]),
        ],
        align='center',
        gap=1
    )
    return stack

slide()